# Narrative Intro
[idk what I'm doing yet]

## Import libraries

In [3]:
from STA554_proj2_pt1 import SparkDataCheck 

In [4]:
# Code to reimport class for debugging
import importlib
import STA554_proj2_pt1
importlib.reload(STA554_proj2_pt1)

from STA554_proj2_pt1 import SparkDataCheck

In [5]:
# Initialize spark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("air").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 13:52:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read in data using our module

In [6]:
air_csv = SparkDataCheck.from_csv(spark, "air.csv")

print("csv loaded!")

csv loaded!


## 4 - 5 examples of the methods from SparkDataCheck

In [7]:
# Print df head so we can see what cols we have to work with
air_csv.df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-17 21:00:00|   2.2|       137

26/03/17 13:52:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-jgreene4@ncsu.edu/air.csv


# 1. Example usage of `check_numeric_range`

In [8]:
# Check to see if C6H6 values are within an expected range (above 0)
air_csv.check_numeric_range("C6H6(GT)", lower = 0)

# Show output selecting relevant cols for visibility
air_csv.df.select("C6H6(GT)", "C6H6(GT)_valid_range").show()

+--------+--------------------+
|C6H6(GT)|C6H6(GT)_valid_range|
+--------+--------------------+
|    11.9|                true|
|     9.4|                true|
|     9.0|                true|
|     9.2|                true|
|     6.5|                true|
|     4.7|                true|
|     3.6|                true|
|     3.3|                true|
|     2.3|                true|
|     1.7|                true|
|     1.3|                true|
|     1.1|                true|
|     1.6|                true|
|     3.2|                true|
|     8.0|                true|
|     9.5|                true|
|     6.3|                true|
|     5.0|                true|
|     5.2|                true|
|     7.3|                true|
+--------+--------------------+
only showing top 20 rows


# 2. Example incorrect usage of `min_max_summary`

In [9]:
# Example where a non-numeric col is provided to the min_max_summary method (returns message)
air_csv.min_max_summary("Date")

Column 'Date' is not numeric


# 3. Example correct usage of `min_max_summary`

In [10]:
# Now do min_max_summary on a numeric column (C6H6) 
air_csv.min_max_summary("C6H6(GT)")

,min,max
0,-200.0,63.7


That reminds us that the NAs in this CSV are -200 and we need to do some data cleaning to fix it, let's explore that first

# 4. Example usage of `check_missing` and `count_levels_bool` (the boolean version of counts associated with string columns)

In [11]:
air_csv.df = air_csv.df.replace(-200, None)

air_csv.check_missing("C6H6(GT)")

air_csv.df.select("C6H6(GT)", "C6H6(GT)_is_na").show()

air_csv.count_levels_bool("C6H6(GT)_is_na")

+--------+--------------+
|C6H6(GT)|C6H6(GT)_is_na|
+--------+--------------+
|    11.9|         false|
|     9.4|         false|
|     9.0|         false|
|     9.2|         false|
|     6.5|         false|
|     4.7|         false|
|     3.6|         false|
|     3.3|         false|
|     2.3|         false|
|     1.7|         false|
|     1.3|         false|
|     1.1|         false|
|     1.6|         false|
|     3.2|         false|
|     8.0|         false|
|     9.5|         false|
|     6.3|         false|
|     5.0|         false|
|     5.2|         false|
|     7.3|         false|
+--------+--------------+
only showing top 20 rows


,C6H6(GT)_is_na,count
0,True,366
1,False,8991


# 5. Example usage of `min_max_summary` with a group

In [12]:
# Convert -200s to NA values
air_csv.df = air_csv.df.replace(-200, None)

# Now do min_max_summary on a numeric column (C6H6) grouped by date with the NAs properly encoded (ignored)
air_csv.min_max_summary("C6H6(GT)", "Date")

,Date,min,max
0,9/2/2004,3.1,30.0
1,12/26/2004,0.7,14.0
2,2/18/2005,0.7,17.6
3,10/10/2004,3.6,17.2
4,10/11/2004,1.4,25.6
...,...,...,...
386,1/23/2005,1.6,10.1
387,6/28/2004,2.7,32.4
388,8/16/2004,1.5,12.0
389,12/20/2004,0.7,11.9


# air.csv as a `pandas` df

In [15]:
import pandas as pd

air_pd = pd.read_csv("air.csv")
air_csv_pd = SparkDataCheck.from_pandas(spark, air_pd)

# 1 example using a method on the `pandas` object

In [16]:
# Swap NAs
air_csv_pd.df = air_csv_pd.df.replace(-200, None)

# Check min max by date of a different gas
air_csv_pd.min_max_summary("NOx(GT)", "Date")

,Date,min,max
0,3/11/2004,16.0,383.0
1,3/12/2004,21.0,340.0
2,3/10/2004,89.0,172.0
3,3/13/2004,53.0,296.0
4,3/15/2004,66.0,478.0
...,...,...,...
386,3/30/2005,82.0,475.0
387,3/31/2005,73.0,673.0
388,4/3/2005,60.0,367.0
389,4/4/2005,52.0,594.0


# Part 2